<a href="https://colab.research.google.com/github/MatthewFaiello/MLsessions/blob/main/ISEA_Week5_ML_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Your Task: Predict Student Success

The purpose of this HW is to get you hands on with real data trying out the modelling techniques we talked about.

You are free to use gen-ai with this project to help with the coding (of course, you don't have to!). [Hands on Machine Learning](https://www.oreilly.com/library/view/hands-on-machine-learning/9781098125967/) is also a great resource.

Your code needs to run, but I want you to focus less on the specifics of the code and more on understanding the component steps that go into building and validating a model. Creating code is now pretty easy, creating a "good" model is hard.

For this exercise we will use open data on student dropout from Portugal. Full documentation is available [here](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)

M.V.Martins, D. Tolledo, J. Machado, L. M.T. Baptista, V.Realinho. (2021) "Early prediction of student’s performance in higher education: a case study" Trends and Applications in Information Systems and Technologies, vol.1, in Advances in Intelligent Systems and Computing series. Springer. DOI: 10.1007/978-3-030-72657-7_16

You will turn in on the class website a google slide deck that has:
1. A cover slide contianing your name (and all group member names if working together) and a link to your colab (**Create slide 1 now**)
2. 3 slides answering the questions below - they are clearly indicated as you go through the colab notebook.


# Get the data

Here I provide some code to get the data for you

In [65]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

import pandas as pd

from sklearn.model_selection import train_test_split

import numpy as np

In [58]:
# fetch dataset
predict_students_dropout_and_academic_success = fetch_ucirepo(id=697)

# data (as pandas dataframes)
X = predict_students_dropout_and_academic_success.data.features
y = predict_students_dropout_and_academic_success.data.targets
df = df = X.join(y)

# metadata
print(predict_students_dropout_and_academic_success.metadata)

# variable information
print(predict_students_dropout_and_academic_success.variables)

                                              name     role         type  \
0                                   Marital Status  Feature      Integer   
1                                 Application mode  Feature      Integer   
2                                Application order  Feature      Integer   
3                                           Course  Feature      Integer   
4                       Daytime/evening attendance  Feature      Integer   
5                           Previous qualification  Feature      Integer   
6                   Previous qualification (grade)  Feature   Continuous   
7                                      Nacionality  Feature      Integer   
8                           Mother's qualification  Feature      Integer   
9                           Father's qualification  Feature      Integer   
10                             Mother's occupation  Feature      Integer   
11                             Father's occupation  Feature      Integer   
12          

# 1 Data Checking

- Look at your outcome variable - any cases to exclude?
- Determine the base-rate accuracy for a naive model
- Create Test and Training Sets
- Look at distributions of x variables, look up meta data, decide which to include

At the end of this section you should have
`x_train`, `x_text`, `y_train`, `y_test`
And an estimate of the base rate accuracy.

In [73]:
# check target distribution
pd.DataFrame({"count": df["Target"].value_counts(dropna=False),
              "percent": df["Target"].value_counts(normalize=True, dropna=False).mul(100).round(2)})

# recode so target is either "Graduate" or "Other"
df["Target_"] = df["Target"].replace({"Dropout": "Other", "Enrolled": "Other"})

# target base-rate
pd.DataFrame({"count": df["Target_"].value_counts(dropna=False),
              "percent": df["Target_"].value_counts(normalize=True, dropna=False).mul(100).round(2)})

# stratified test and train sets
train_df, test_df = train_test_split(df,
                                     test_size=0.2,
                                     random_state=42,
                                     stratify=df["Target_"])
# check stratification
pd.DataFrame({"train": train_df["Target_"].value_counts(normalize=True, dropna=False).mul(100).round(2),
              "test": test_df["Target_"].value_counts(normalize=True, dropna=False).mul(100).round(2)})

# check features
metaData = predict_students_dropout_and_academic_success.variables; metaData
num = df.select_dtypes(include="number")
pd.DataFrame({"mean": num.mean(),
              "median": num.median(),
              "mode": num.mode(dropna=True).iloc[0]})

# select features
range_cols = df.iloc[:, np.r_[4, 6, 12:32]].columns
df_subset = df.loc[:, range_cols]; df_subset

,Daytime/evening attendance,Previous qualification (grade),Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,...,Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade)
0,1,122.0,127.3,1,0,0,1,1,0,20,...,0,0,0,0.000000,0,0,0,0,0,0.000000
1,1,160.0,142.5,1,0,0,0,1,0,19,...,6,6,6,14.000000,0,0,6,6,6,13.666667
2,1,122.0,124.8,1,0,0,0,1,0,19,...,6,0,0,0.000000,0,0,6,0,0,0.000000
3,1,122.0,119.6,1,0,0,1,0,0,20,...,6,8,6,13.428571,0,0,6,10,5,12.400000
4,0,100.0,141.5,0,0,0,1,0,0,45,...,6,9,5,12.333333,0,0,6,6,6,13.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4419,1,125.0,122.2,0,0,0,1,1,0,19,...,6,7,5,13.600000,0,0,6,8,5,12.666667
4420,1,120.0,119.0,1,0,1,0,0,0,18,...,6,6,6,12.000000,0,0,6,6,2,11.000000
4421,1,154.0,149.5,1,0,0,1,0,1,30,...,7,8,7,14.912500,0,0,8,9,1,13.500000
4422,1,180.0,153.8,1,0,0,1,0,1,20,...,5,5,5,13.800000,0,0,5,6,5,12.000000


# 2 Train a Model
* Pick one of the models we discussed today and train it.
* Report its accuracy and print a confusion matrix.
   * How much better is your model than the base rate?
   * How does accuracy on the train set compare to accuracy on the test set?
   * **Report Slide 2: Include an image of the confusion matrix, the base rate accuracy, train-set accuracy and test-set accuracy**

# 3 Train a Different Model
* Repeat all the steps in 2, but use a different model
* In addition, compare the accuracy of 1 and 2
* **Report Slide 3: Model 2 confusion matrix, train-set accuracy and test-set accuracy. Comparison Model 1 and Model 2 accuracy**

# 4 Reflection
* **Type responses on Slide 4**
* Contextualizing accuracy - think about different use cases for your model, which ones would you feel its accurate enough to use for? I only asked you to look at overall accuracy, is that good enough?
* Contextualizing features - think about these same use cases, are the prediction features you included appropriate for these uses?
* Generalizability - again thinking about your features, could you use this model in other educational contexts? How hard would it be to get that same data? Are there issues with it generalizing over time and location?

# 5 Extra Credit
* Consider ensembling your two models. Does that perform better?
* Check accuracy for different subgroups